# Task 1 — Classification (§2.1 BatchNorm · §2.2 Dropout · §2.4 Feature Maps)

**Kaggle setup:** Accelerator → GPU T4x2 · Internet ON

**After this notebook you will have:**
- `checkpoints/classifier.pth` saved to Kaggle output
- W&B runs logged for §2.1 and §2.2
- `CLASSIFIER_DRIVE_ID` updated in `models/multitask.py` and pushed to GitHub
- Ready to submit **4.1b / 4.1c** to Gradescope

## 1 · Setup

In [ ]:
import os

# ── W&B API key ────────────────────────────────────────────────────────────────
os.environ["WANDB_API_KEY"] = "PASTE_YOUR_WANDB_API_KEY_HERE"

# ── GitHub token for pushing at the end ───────────────────────────────────────
# Create one at: https://github.com/settings/tokens (needs 'repo' scope)
GITHUB_TOKEN    = "PASTE_YOUR_GITHUB_TOKEN_HERE"
GITHUB_USERNAME = "usnaveen"
GITHUB_REPO     = "A2_Deep_Learning"

# ── Clone repo ─────────────────────────────────────────────────────────────────
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git da6401_assignment_2
%cd da6401_assignment_2

!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

## 2 · Download Dataset

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")

## 3 · §2.2 — Dropout Experiment
Trains 3 runs (p=0, 0.2, 0.5) with improved augmentation + label smoothing.

In [ ]:
!python train.py --task classify --experiment exp-dropout --dropout 0.0 --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2
!python train.py --task classify --experiment exp-dropout --dropout 0.2 --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2
!python train.py --task classify --experiment exp-dropout --dropout 0.5 --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2

## 4 · §2.1 — BatchNorm Ablation
Trains **without** BatchNorm so you can compare activation distributions in W&B.

In [ ]:
!python train.py --task classify --experiment exp-batchnorm --no-bn --dropout 0.5 --epochs 120 --lr 1e-4 --batch-size 32 --num-workers 2

## 5 · §2.4 — Feature Map Visualization

In [ ]:
!python inference.py --mode featuremaps --num-workers 2

## 6 · Check Best Checkpoint F1
Verify the best model before uploading.

In [ ]:
import os, torch
from sklearn.metrics import f1_score
from data.pets_dataset import create_dataloaders
from models.classification import VGG11Classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, val_loader, _ = create_dataloaders(root="./data/oxford_pet", batch_size=64, num_workers=2)

model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
model.load_state_dict(torch.load("checkpoints/classifier.pth", map_location=device, weights_only=False))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        logits = model(batch["image"].to(device))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(batch["label"].tolist())

f1 = f1_score(all_labels, all_preds, average="macro")
print(f"\n✅  Val Macro-F1 = {f1:.4f}")
print("   (need > 0.50 for 4.1b · > 0.80 for 4.1c)")

## 7 · Upload `classifier.pth` to Google Drive

**Manual steps (takes ~2 minutes):**
1. In the Kaggle right sidebar, click **Output** → download `checkpoints/classifier.pth`
2. Go to [drive.google.com](https://drive.google.com) → upload `classifier.pth`
3. Right-click the file → **Share** → **Anyone with the link** → Copy link
4. The link looks like: `https://drive.google.com/file/d/`**`FILE_ID`**`/view`
5. Paste just the **FILE_ID** part into the cell below and run it

In [ ]:
# ── Paste your Google Drive file ID here ──────────────────────────────────────
CLASSIFIER_DRIVE_ID = "PASTE_FILE_ID_HERE"   # ← only the ID, not the full URL
# ─────────────────────────────────────────────────────────────────────────────

multitask_path = "models/multitask.py"
with open(multitask_path) as f:
    content = f.read()

# Replace the placeholder with the real ID
import re
content = re.sub(
    r'CLASSIFIER_DRIVE_ID\s*=\s*"[^"]*"',
    f'CLASSIFIER_DRIVE_ID = "{CLASSIFIER_DRIVE_ID}"',
    content
)

with open(multitask_path, "w") as f:
    f.write(content)

print(f"✅  Updated models/multitask.py → CLASSIFIER_DRIVE_ID = '{CLASSIFIER_DRIVE_ID}'")

# Quick sanity check — verify gdown can fetch it
import gdown, tempfile, os
with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as tmp:
    tmp_path = tmp.name
try:
    gdown.download(id=CLASSIFIER_DRIVE_ID, output=tmp_path, quiet=False, fuzzy=True)
    size_mb = os.path.getsize(tmp_path) / 1e6
    print(f"✅  Download test passed ({size_mb:.1f} MB) — Drive sharing is correct")
except Exception as e:
    print(f"❌  Download test FAILED: {e}")
    print("    → Make sure the file is shared as 'Anyone with the link'")
finally:
    os.unlink(tmp_path)

## 8 · Push to GitHub → Submit to Gradescope

In [ ]:
import subprocess

# Configure git identity (required for commits)
!git config user.email "naveen@student.iitm.ac.in"
!git config user.name  "Naveen"

# Stage only multitask.py (the only file that changed for Gradescope)
!git add models/multitask.py
!git diff --cached --stat

!git commit -m "task1: set CLASSIFIER_DRIVE_ID for Gradescope submission"
!git push

print("\n🎉  Done! You can now submit to Gradescope.")
print("   Gradescope will test: 4.1b (F1 > 0.5) and 4.1c (F1 > 0.8)")